<a href="https://colab.research.google.com/github/J-K-O-Dev/Large-Language-Model-CS25/blob/main/Week4_work/GPT2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers datasets accelerate

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# Changed to AutoModelForCausalLM and removed num_labels
model = AutoModelForCausalLM.from_pretrained("gpt2")
model.config.pad_token_id = model.config.eos_token_id

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [3]:
from datasets import load_dataset

# 1. Load the MBPP dataset
dsa_dataset = load_dataset("google-research-datasets/mbpp", split="train")

# 2. Define the formatting function for GPT-2
def format_mbpp_prompt(example):
    # Concatenate the problem description and the solution
    formatted_text = f"/* Problem:\n{example['text']} */\n\n// Solution:\n{example['code']}"

    # Return it as a new 'text' column
    return {"text": formatted_text}

# 3. Map the function across the dataset
ready_to_tokenize_dataset = dsa_dataset.map(format_mbpp_prompt)

# Check the first formatted example to ensure it looks right
print(ready_to_tokenize_dataset[0]['text'])

README.md:   0%|          | 0.00/9.06k [00:00<?, ?B/s]

full/train-00000-of-00001.parquet:   0%|          | 0.00/87.2k [00:00<?, ?B/s]

full/test-00000-of-00001.parquet:   0%|          | 0.00/116k [00:00<?, ?B/s]

full/validation-00000-of-00001.parquet:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

full/prompt-00000-of-00001.parquet:   0%|          | 0.00/7.88k [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/90 [00:00<?, ? examples/s]

Generating prompt split:   0%|          | 0/10 [00:00<?, ? examples/s]

Map:   0%|          | 0/374 [00:00<?, ? examples/s]

/* Problem:
Write a function to find the longest chain which can be formed from the given set of pairs. */

// Solution:
class Pair(object): 
	def __init__(self, a, b): 
		self.a = a 
		self.b = b 
def max_chain_length(arr, n): 
	max = 0
	mcl = [1 for i in range(n)] 
	for i in range(1, n): 
		for j in range(0, i): 
			if (arr[i].a > arr[j].b and
				mcl[i] < mcl[j] + 1): 
				mcl[i] = mcl[j] + 1
	for i in range(n): 
		if (max < mcl[i]): 
			max = mcl[i] 
	return max


In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Define a tokenization function
def tokenize_function(examples):
    # Ensure truncation is handled if your texts can be longer than the model's max input length
    # The DataCollatorForLanguageModeling will handle padding and label creation.
    tokenized_inputs = tokenizer(
        examples["text"],
        truncation=True,
        max_length=tokenizer.model_max_length # Use the model's max length for truncation
    )
    return tokenized_inputs

# Apply the tokenization function to the dataset using .map()
tokenized_dataset = ready_to_tokenize_dataset.map(tokenize_function, batched=True)

training_args = TrainingArguments(
    output_dir="./gpt2_coding_model",
    learning_rate=0.01,
    per_device_train_batch_size=8,
    num_train_epochs=3,                  # How many times it reads the entire dataset
    weight_decay=0.01,
    logging_steps=1,
    report_to="none",
    optim="adamw_torch" # Changed optimizer to a valid option
)

# Initialize the DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False # Set to False for Causal Language Modeling (decoder-only models)
)

trainer = Trainer(
    model=model,
    args=training_args,
    # Use the tokenized dataset directly, as it's already the 'train' split
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    # validation_dataset=tokenized_dataset['validation'],
    # test_dataset=tokenized_dataset['test']
)

# Start the training loop!
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
1,6.932137
2,15.655659
3,10.528006
4,8.722286
5,9.545039
6,8.190910
7,7.423017
8,7.106371
9,6.535539
10,7.770006


In [ ]:
# Save the trained model
trainer.save_model("./final_gpt2_coding_model")
print("Model saved to ./final_gpt2_coding_model")